<a href="https://colab.research.google.com/github/alj-x64/Realtime-Bass-Tablature-Transcription/blob/main/Adaptive_Multi_Stage_HPO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import random
import math
import copy
import os
import csv
import pickle # Idinagdag para sa Checkpointing and Persistence

class AdaptiveMultiStageStressTestedHPO:
    def __init__ (self, 
                  budgetTrial, 
                  alpha,   #accuracy priority
                  beta,    #latency priority
                  rho_S,   #selection stage budget trial
                  rho_R,   #refinement stage budget trial
                  gamma,   #allocation ratio
                  logfile,
                  checkpoint_file="hpo_state.pkl" # Bagong parameter para sa checkpoint
                 ):
        
        self.B = budgetTrial
        self.alpha = alpha
        self.beta = beta

        self.S = int(rho_S * self.B)
        self.R = int(rho_R * self.B)
        self.gamma_alloc = gamma

        self.P = max(2, math.ceil(self.gamma_alloc * self.S))
        self.G = max(1, math.floor(self.S / self.P))

        print(f"--- HPO Budgets Initialized ---")
        print(f"Total Budget (B): {self.B} trials")
        print(f"Selection Budget (S): {self.S} | Refinement Budget (R): {self.R}")
        print(f"Population Size (P): {self.P} | Generations (G): {self.G}")
        print(f"-------------------------------")

        self.search_space = {
            'learning_rate': {'type': 'continuous', 'range': [1e-5, 1e-2]},
            'dropout_rate': {'type': 'continuous', 'range' : [0.1, 0.5]},
            'activation' : {'type': 'categorical', 'values': ['ReLU', 'Tanh', 'ELU']},
            'conv_layers': {'type': 'discrete', 'range': [1, 4]},
            'filter_layers': {'type': 'categorical', 'values': [16, 32, 64, 128]},
            'kernel_size': {'type': 'categorical', 'values': [2, 3, 5, 7]}
        }
        self.logfile = logfile
        self.checkpoint_file = checkpoint_file
        self._init_csv_logger()

    def _init_csv_logger(self):
        file_exists = os.path.isfile(self.logfile)
        if not file_exists:
            with open(self.logfile, mode='w', newline='') as f:
                writer = csv.writer(f)
                writer.writerow(['Stage', 'Gen or Rank', 'Learning Rate', 'Activation Function',
                                 '# of Convolution Layers', '# of Filter Layers', "Kernel Size",
                                 'Loss Value', 'Latency', 'Fitness Score', 'Status'])

    def individual_logging(self, stage, gen_rank, config, loss, latency, fitness, status):
        with open(self.logfile, mode='a', newline='') as f:
            writer = csv.writer(f)
            writer.writerow([stage, gen_rank, f"{config['learning_rate']:.6f}", config['dropout_rate'],
                             config['activation'], config['conv_layers'], config['filter_layers'],
                             config['kernel_size'], f"{loss:.4f}", f"{latency:.2f}", f"{fitness:.4f}", status])
            
    # ========================================================
    # CHECKPOINTING MECHANISMS (Bagong Dagdag, beh)
    # ========================================================
    def save_checkpoint(self, state):
        """Sine-save ang buong state ng HPO sa disk (Hard Drive/Google Drive)"""
        with open(self.checkpoint_file, 'wb') as f:
            pickle.dump(state, f)
        print(f"[CHECKPOINT SAVED] Nai-save ang kasalukuyang progress sa {self.checkpoint_file}")

    def load_checkpoint(self):
        """Bumubuhay ng lumang state kung sakaling namatay ang Colab"""
        if os.path.exists(self.checkpoint_file):
            with open(self.checkpoint_file, 'rb') as f:
                state = pickle.load(f)
            print(f"[CHECKPOINT LOADED] Ipinagpapatuloy ang na-save na progress...")
            return state
        return None

    def clear_checkpoint(self):
        """Tinatanggal ang checkpoint file kapag tapos na ang buong thesis execution"""
        if os.path.exists(self.checkpoint_file):
            os.remove(self.checkpoint_file)
            print("[CHECKPOINT CLEARED] Tapos na ang HPO. Nilinis na ang memory file.")

    def individual_generation (self):
        individual = {}
        for key, parameter in self.search_space.items():
            if parameter['type'] == 'continuous':
                if key == 'learning_rate':
                    individual[key] = 10 ** np.random.uniform(np.log10(parameter['range'][0]), np.log10(parameter['range'][1]))
                else:
                    individual[key] = np.random.uniform(parameter['range'][0], parameter['range'][1])
            elif parameter['type'] == 'discrete':
                individual[key] = np.random.randint(parameter['range'][0], parameter['range'][1] + 1)
            elif parameter['type'] == 'categorical':
                individual[key] = random.choice(parameter['values'])
        return individual
    
    def fitness_function (self, loss, latency):
        latency_penalty = (latency/200) if latency > 0 else 0
        return self.alpha * (1 - loss) - self.beta * (latency_penalty)

    def crossover (self, parent1, parent2):
        offspring = {}
        for key in self.search_space.keys():
            offspring[key] = parent1[key] if random.random() < 0.5 else parent2[key]
        return offspring
    
    def mutate (self, individual):
        mutated = copy.deepcopy(individual)
        for key, parameter in self.search_space.items():
            if random.random() < 0.5:
                if parameter['type'] == 'continuous':            
                    sigma = (parameter['range'][1] - parameter['range'][0]) * 0.1
                    new_val = mutated[key] + np.random.normal(0, sigma)
                    mutated[key] = np.clip(new_val, parameter['range'][0], parameter['range'][1])
                elif parameter['type'] == 'discrete':
                    mutated[key] = np.random.randint(parameter['range'][0], parameter['range'][1] + 1)
                elif parameter['type'] == 'categorical':
                    mutated[key] = random.choice(parameter['values'])
        return mutated
    
    def optimization (self, train_eval):
        # I-check kung may nakaimbak na state bago magsimula
        state = self.load_checkpoint()
        
        if state:
            # RESUME PREVIOUS STATE
            stage = state['stage']
            population = state['population']
            total_loss = state['total_loss']
            start_gen = state.get('generation', 0)
            refine_rank = state.get('refine_rank', 0)
        else:
            # START FROM SCRATCH
            print(f"Entering selection stage")
            stage = 'selection'
            population = [{'config': self.individual_generation(), 'fitness': 0, 'latency': 0, 'loss': 0} for _ in range(self.P)]
            total_loss = []
            start_gen = 0
            refine_rank = 0

        # ===============================================
        # SELECTION STAGE
        # ===============================================
        if stage == 'selection':
            for generations in range(start_gen, self.G):
                for ind in population:
                    # Wag na i-evaluate ulit kung na-evaluate na (fitness > 0)
                    if ind['fitness'] == 0:
                        loss, latency = train_eval(ind['config'], stress_test=False)
                        ind['loss'] = loss
                        ind['latency'] = latency
                        ind['fitness'] = self.fitness_function(loss, latency)
                        total_loss.append(loss)
                        
                        self.individual_logging("Selection", f"Generation {generations + 1}", 
                                                ind['config'], loss, latency, ind['fitness'], "Evaluated")
                        
                population.sort(key=lambda x:x['fitness'], reverse=True)
                survivors = population[:max(1, len(population)//2)]

                # Crossover and Mutation logic
                next_gen = copy.deepcopy(survivors)
                while len(next_gen) < self.P:
                    p1, p2 = random.sample(survivors, 2) if len(survivors) > 1 else (survivors[0], survivors[0])
                    offspring = self.crossover(p1['config'], p2['config'])
                    mutated = self.mutate(offspring)
                    next_gen.append({'config': mutated, 'fitness': 0, 'latency': 0, 'loss': 0})

                population = next_gen
                
                # UPDATE CHECKPOINT EVERY GENERATION
                self.save_checkpoint({
                    'stage': 'selection',
                    'generation': generations + 1,
                    'population': population,
                    'total_loss': total_loss
                })

            # Tapos na ang selection, transition to refinement state
            stage = 'refinement'
            
            # I-evaluate yung final generation
            for ind in population:
                if ind['fitness'] == 0:
                    loss, latency = train_eval(ind['config'], stress_test=False)
                    ind['loss'] , ind['latency'] = loss, latency
                    ind['fitness'] = self.fitness_function(loss, latency)
                    total_loss.append(loss)

            population.sort(key=lambda x: x['fitness'], reverse=True)
            self.save_checkpoint({
                'stage': 'refinement',
                'population': population,
                'total_loss': total_loss,
                'refine_rank': 0
            })

        # ===============================================
        # REFINEMENT STAGE
        # ===============================================
        if stage == 'refinement':
            p25_loss = np.percentile(total_loss, 25) 
            
            for rank in range(refine_rank, min(self.R, len(population))):
                candidate = population[rank]
                print(f"Testing candidate {rank + 1} (Fitness score: {candidate['fitness']:.4f})")

                # ROBUST-BASED STRESS TESTING
                stress_loss, stress_latency = train_eval(candidate['config'], stress_test=True)

                # LATENCY JITTER
                p_trigger = 0.10
                if np.random.rand() < p_trigger:
                    jitter = np.random.uniform(0, 50)
                    stress_latency += jitter
                    print(f"Added {jitter:.2f}ms latency jitter")

                # UPDATE CHECKPOINT PARA SA NEXT CANDIDATE
                self.save_checkpoint({
                    'stage': 'refinement',
                    'population': population,
                    'total_loss': total_loss,
                    'refine_rank': rank + 1
                })

                # CONSTRAINT-AWARE REFINEMENT
                if stress_latency <= 200.0 and stress_loss <= p25_loss:
                    print(f"ACCEPT THE INDIVIDUAL as the optimal hyperparameter")
                    self.individual_logging("Refinement", f"Rank {rank + 1}", candidate['config'], 
                                            stress_loss, stress_latency, candidate['fitness'], "Promoted as Optimal")
                    self.clear_checkpoint() # Linisin pag nakahanap na ng panalo
                    return candidate['config']
                else:
                    print(f"Individual failed the test. Kill the individual. Promote second best individual to refinement stage")

        print("Refinement budget trial used up. No candidate passed")
        self.individual_logging("Refinement", "Fallback", population[0]['config'], 
                                population[0]['loss'], population[0]['latency'], population[0]['fitness'], "Fallback")
        self.clear_checkpoint() # Linisin kahit fallback ang panalo
        return population[0]['config']